* 比较不同组合组合优化器在不同规模问题上的性能；

* 下面的结果主要比较``alphamind``和``python``中其他优化器的性能差别，我们将尽可能使用``cvxopt``中的优化器，其次选择``scipy``；

* 由于``scipy``在``ashare_ex``上面性能太差，所以一般忽略``scipy``在这个股票池上的表现；

* 时间单位都是毫秒。

* 请在环境变量中设置`DB_URI`指向数据库

In [ ]:
import time


## 0. 数据准备
------------------

In [ ]:
import numpy as np
import pyarrow as pa


## 1. 线性优化（带线性限制条件）
---------------------------------

In [ ]:
from alphamind.portfolio.linearbuilder import linear_builder


In [ ]:
from alphamind.portfolio.meanvariancebuilder import mean_variance_builder

rng = np.random.default_rng(42)


In [ ]:
n = 30


## 2. 线性优化（带L1限制条件）
-----------------------

In [ ]:
er = rng.normal(0.0, 0.01, size=n)


In [ ]:
benchmark = np.full(n, 1.0 / n)
risk = np.ones((n, 1))


## 3. Mean - Variance 优化 （无约束）
-----------------------

In [ ]:
target = (np.array([1.0]), np.array([1.0]))


In [ ]:
covariance = np.eye(n) * 0.02


## 4. Mean - Variance 优化 （Box约束）
---------------

In [ ]:
risk_model = {"cov": covariance, "factor_cov": None, "factor_loading": None, "idsync": None}
started = time.perf_counter()


In [ ]:
linear_status, _, linear_weights = linear_builder(er, 0.0, 1.0, risk, target)


## 5. Mean - Variance 优化 （Box约束以及线性约束）
----------------

In [ ]:
linear_seconds = time.perf_counter() - started


In [ ]:
started = time.perf_counter()
mv_status, _, mv_weights = mean_variance_builder(
    er, risk_model, benchmark, 0.0, 1.0, risk, target
)


## 6. 线性优化（带二次限制条件）
-------------------------

In [ ]:
mv_seconds = time.perf_counter() - started


In [ ]:
assert linear_status in ("optimal", "optimal_inaccurate")
assert mv_status in ("optimal", "optimal_inaccurate")


In [ ]:
pa.table({"method": ["linear", "mean_variance"], "seconds": [linear_seconds, mv_seconds]})
